In [6]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer


from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV



# Problem 1: tweet classification - Trudeau vs Trump

In [2]:
# load the data
url = url = 'https://raw.githubusercontent.com/um-perez-alvaro/Data-Science-Practice/master/Data/twitter.csv'
data = pd.read_csv(url)
data.head()

,timestamp,text,user
0,2020-03-02 23:06:03,"WOW! Thank you, just landed, see everyone soon...",realDonaldTrump
1,2020-03-02 21:47:49,Departing for the Great State of North Carolin...,realDonaldTrump
2,2020-03-02 21:32:54,They are staging a coup against Bernie!,realDonaldTrump
3,2020-03-02 19:55:40,THANK YOU!https://www.breitbart.com/tech/2020/...,realDonaldTrump
4,2020-03-02 19:55:07,Michelle @FischbachMN7 is running for Congress...,realDonaldTrump


This is a corpus of tweets from Donald Trump and Justin Trudeau. 
The **goal** is to build a classification pipeline that predicts the author (Trump or Trudeau) of a tweet based on the text.

**Part 1:** Define the feature matrix X and the target vector y from the dataframe, and then split X and y into training and testing sets.

In [4]:
X = data.text
y = data.user



# train/test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y)


**Part 2:** build a classification pipeline (count vectorizer + Naive Bayes model), and fit the pipeline to the training data.

In [7]:
pipe = Pipeline(steps=[
    ('vect', CountVectorizer()), 
    ('clf', MultinomialNB()) 
])

pipe.fit(X_train,y_train)

Pipeline(steps=[('vect', CountVectorizer()), ('clf', MultinomialNB())])

**Part 3:** Evaluate the performance of your classification pipeline on the test set

In [8]:
y_test_pred = pipe.predict(X_test)

In [9]:


confusion_matrix(y_test,y_test_pred)



array([[ 95,   1],
       [  7, 162]])

In [10]:
accuracy_score(y_test, y_test_pred)

0.969811320754717

A 97% accurate evaluation is quite strong; I would maybe be worried about overfitting the dataset here actually.

**Part 4:** What words does the model use to choose between Trump or Trudeau

In [11]:
# store the vocabulary of X_train
words = pipe['vect'].get_feature_names_out()
pipe['clf'].classes_

array(['JustinTrudeau', 'realDonaldTrump'], dtype='<U15')

In [12]:

trudeau_word_count = pipe['clf'].feature_count_[0,:]

trump_word_count = pipe['clf'].feature_count_[1,:]

In [13]:

words = pd.DataFrame({'word' : words, 'Trudeau' : trudeau_word_count, 'Trump' : trump_word_count}).set_index('word')
words.head()



,Trudeau,Trump
word,,
00,0.0,10.0
000,12.0,9.0
01,1.0,0.0
02,1.0,0.0
03,36.0,1.0


In [15]:
# convert the counts into frequencies
words.Trudeau = words.Trudeau/words.Trudeau.sum()
words.Trump = words.Trump/words.Trump.sum()
words.head()

,Trudeau,Trump
word,,
00,0.000000,0.000704
000,0.000869,0.000634
01,0.000072,0.000000
02,0.000072,0.000000
03,0.002606,0.000070


In [17]:
# ratios
words['trudeau_ratio'] = words.Trudeau/words.Trump
words['trump_ratio'] = words.Trump/words.Trudeau

In [18]:


words.sort_values(by='trump_ratio', ascending=False).head(20)



,Trudeau,Trump,trudeau_ratio,trump_ratio
word,,,,
00,0.0,0.000704,0.0,inf
lightweight,0.0,0.000070,0.0,inf
likely,0.0,0.000070,0.0,inf
linda,0.0,0.000070,0.0,inf
lindasuhler,0.0,0.000070,0.0,inf
lindsey,0.0,0.000070,0.0,inf
lindseygrahamsc,0.0,0.000070,0.0,inf
lisamarieboothe,0.0,0.000070,0.0,inf
list,0.0,0.000070,0.0,inf


In [19]:


words.sort_values(by='trudeau_ratio', ascending=False).head(20)



,Trudeau,Trump,trudeau_ratio,trump_ratio
word,,,,
personnel,0.000072,0.0,inf,0.0
ted,0.000072,0.0,inf,0.0
meals,0.000145,0.0,inf,0.0
teamgushue,0.000072,0.0,inf,0.0
joining,0.000072,0.0,inf,0.0
gloves,0.000072,0.0,inf,0.0
consider,0.000072,0.0,inf,0.0
technologies,0.000072,0.0,inf,0.0
secondharvestca,0.000072,0.0,inf,0.0


**Bonus:** can you write a Trump or Trudeau tweet?

Trudeau: We are sending personnel, meals, and condolences to those across the province.